In [ ]:
# Load packages
import os
import shlex
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

# PHENOTYPE FILE

In [ ]:
# Define path to workspace bucket 
bucket=f"{os.environ['WORKSPACE_BUCKET']}/data/saige_phenotype_files/aml_biosample_age_files/EUR_phenotype_file.txt"
# Load phenotype file  
df = pd.read_csv(bucket, sep='\t')
# Format person id to BGEN format 
df['person_id'] = df['person_id'].astype(str) + '_' + df['person_id'].astype(str)
# Save as phenotype file to workspace bucket 
df.to_csv(f"EUR_phenotype_file.txt", 
          sep='\t',
          index=None)

In [ ]:
# Save phenotype file to workspace bucket 
!gcloud storage cp EUR_phenotype_file.txt {os.environ['WORKSPACE_BUCKET']}/data/staar/phenotype/

# COMMAND 

In [ ]:
%%writefile STAARpipeline_Null_Model.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read command-line arguments
args <- commandArgs(trailingOnly = TRUE)

# Define input/output file paths
phenotype_path <- args[1]
grm_path <- args[2]
output_path <- args[3]

# Print input/output paths
cat("PHENOTYPE:", phenotype_path, "\n")
cat("GENETIC RELATIONSHIP MATRIX:", grm_path, "\n")
cat("OUTPUT PATH:", output_path, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

# load packages 
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)
library(STAAR)
library(STAARpipeline)

# Read phenotype/covariate data
phenotype <- read.delim(
    phenotype_path,
    header = TRUE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# Load the sparse genetic relationship matrix
grm <- get(load(grm_path))

# Check that the input data loaded correctly
print(head(phenotype))
print(head(grm))

###########################################################
# FIT NULL MODEL
###########################################################

# Fit the Gaussian null model using the sparse GRM
obj_nullmodel <- fit_null_glmmkin_Binary_SPA(
    aml_biosample_age ~ age + sex + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10,
    data = phenotype,
    kins = grm,
    use_sparse = TRUE,
    kins_cutoff = 0.022,
    id = "person_id",
    family = binomial(link = "logit"),
    verbose = TRUE
)

# Save the fitted null model to the specified output path
save(obj_nullmodel,file=paste0(output_path))

###########################################################
#   
###########################################################

In [ ]:
# Copy script in workspace bucket 
!gcloud storage cp STAARpipeline_Null_Model.R ${WORKSPACE_BUCKET}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/STAARpipeline_Null_Model.R 

# COMMAND SUBMISSION

In [ ]:

# Define path to logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Define path to script
SCRIPT = "STAARpipeline_Null_Model.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

# Define disk size
DISK_SIZE = 10

# Define path to phenotype file 
PHENOTYPE = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/phenotype/EUR_phenotype_file.txt"

# Define path to sparse grm 
GRM = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/grm/sparseGRM.RData"

# Define path to null model 
OUTPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/null_modle/obj_spa_nullmodel.Rdata"

# R script command
R_COMMAND = 'Rscript "$SCRIPT" "$PHENOTYPE" "$GRM" "$OUTPUT"'

# Command for dsub submission 
command = [
    "aou_dsub",
    "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/zilinli/staarpipeline:0.9.7",
    "--boot-disk-size", "5",
    "--disk-size", str(DISK_SIZE),
    "--machine-type", "n1-standard-2",
    "--logging", LOGGING,
    "--input", f"SCRIPT={PATH}",
    "--input", f"PHENOTYPE={PHENOTYPE}",
    "--input", f"GRM={GRM}",
    "--output", f"OUTPUT={OUTPUT}",
    "--command", R_COMMAND,
]

# Run in same shell as source command
dsub_command = subprocess.run(
    f"source ~/aou_dsub.bash && {shlex.join(command)}",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True
)

print(dsub_command.stdout)
print(dsub_command.stderr)